# NB05 — Feature Curation

**Project:** Evolutionary Computation for Sepsis Mortality Prediction
**Input:** `data/processed/features_block3.parquet` (11,164 × 78)
**Output:** `data/processed/features_curated.parquet` (11,164 × 61), `data/processed/feature_config.json`, `results/tables/05_curation_log.csv`

---

### Purpose
Apply all feature curation decisions to the engineered feature matrix (NB03 output),
producing the analysis-ready dataset for all modelling notebooks (NB07–NB14).

### Curation decisions applied

| Decision | Action | Rationale |
|---|---|---|
| B1 — fio2 unit fix | Already applied in NB03 (`fio2_frac`) | Mixed units in raw `apacheApsVar` |
| B2(a) — GCS components | Drop `eyes_miss`, `motor_miss`, `verbal_miss` | Redundant: `gcs_total = eyes + motor + verbal` |
| B2(b) — ABG redundancy | Drop `pao2`, `fio2_frac`, `pao2_miss`, `fio2_frac_miss` | `pf_ratio` encodes both; pao2 ambiguous without FiO₂ context |
| B2(c) — MAP redundancy | Drop `map_min`, `map_max`, `map_min_miss`, `map_max_miss` | `meanbp` (worst episode) + `map_mean` (24h average) sufficient |
| B4 — anomalous records | Assert 0 NaN in `hospital_mortality` | Fixed in NB02: 91 ICU-expired/hospital-alive records excluded |
| B6 — albumin GP exclusion | Retain in `MODELLING_COLS`; exclude from `GP_TERMINALS` | 39.6% imputed at identical median; imputed value dominates GP expressions |
| Leakage columns | Drop `icu_los_days`, `hospital_los_days`, `icu_mortality` | Known only at discharge — temporal leakage at inference time |
| Raw strings | Drop `ethnicity`, `apacheadmissiondx`, `uniquepid` | Encoded as dummies; redundant ID |

### Deliverables

| # | File | Description |
|---|---|---|
| 1 | `features_curated.parquet` | 11,164 × 61 — 3 metadata + 58 `MODELLING_COLS` |
| 2 | `feature_config.json` | `MODELLING_COLS` (58), `GP_TERMINALS` (26), outcome, split key |
| 3 | `05_curation_log.csv` | Every column decision: kept/dropped, reason, missingness tier |

---

## Cell 1 — Load and validate

**Plan.** Load `features_block3.parquet`. Assert cohort integrity (11,164 patients,
0 NaN outcomes, 65 hospitals). This confirms the B4 fix (NB02) and B1 fix (NB03)
are in place before any curation is applied.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json as _json

_nb_dir = Path().resolve()
PROJECT = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA    = PROJECT / "data" / "processed"
TABLES  = PROJECT / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

feat = pd.read_parquet(DATA / "features_block3.parquet")
print(f"Loaded: {feat.shape[0]:,} rows x {feat.shape[1]} columns")

# ── Assertions ────────────────────────────────────────────────────────────────
assert feat.shape[0] == 11164, f"Expected 11,164 rows, got {feat.shape[0]}"
assert feat["hospital_mortality"].isna().sum() == 0, "B4 FAIL: NaN in hospital_mortality"
assert feat["hospitalid"].nunique() == 65, "Expected 65 hospitals"
assert "fio2_frac" in feat.columns, "B1 FAIL: fio2_frac not found"
assert "fio2" not in feat.columns or feat.columns.tolist().index("fio2_frac") >= 0

n_deaths = int(feat["hospital_mortality"].sum())
mort_pct = feat["hospital_mortality"].mean() * 100

print(f"Patients      : {feat.shape[0]:,}")
print(f"Hospitals     : {feat['hospitalid'].nunique()}")
print(f"Deaths        : {n_deaths:,} ({mort_pct:.2f}%)")
print(f"NaN outcome   : {feat['hospital_mortality'].isna().sum()} (expect 0)")
print(f"fio2_frac     : present (B1 fix confirmed)")
print()
print("All assertions passed — ready for curation.")

Loaded: 11,164 rows x 78 columns
Patients      : 11,164
Hospitals     : 65
Deaths        : 1,885 (16.88%)
NaN outcome   : 0 (expect 0)
fio2_frac     : present (B1 fix confirmed)

All assertions passed — ready for curation.


### Findings — Cell 1: Load and validate

---

| Check | Expected | Actual | Status |
|---|---|---|---|
| Rows | 11,164 | **11,164** | PASS |
| Columns | 78 | **78** | PASS |
| Hospitals | 65 | **65** | PASS |
| NaN in `hospital_mortality` | 0 | **0** | PASS — B4 fix confirmed |
| `fio2_frac` present | Yes | **Yes** | PASS — B1 fix confirmed |
| Overall mortality | ~16.88% | **16.88%** | PASS |

---
## Cell 2 — Drop leakage and metadata columns

**Plan.** Remove columns that must not enter any model. Three categories:
- **Leakage:** `icu_los_days`, `hospital_los_days`, `icu_mortality` — known only at discharge
- **Raw strings:** `ethnicity`, `apacheadmissiondx` — already encoded as dummy columns
- **Redundant ID:** `uniquepid` — `patientunitstayid` is the primary key

Assert that none of the post-hoc leakage columns remain after dropping.

In [2]:
LEAKAGE  = ["icu_los_days", "hospital_los_days", "icu_mortality"]
RAW_STR  = ["ethnicity", "apacheadmissiondx"]
REDUND   = ["uniquepid"]
DROP_META = LEAKAGE + RAW_STR + REDUND

df = feat.drop(columns=DROP_META)
print(f"Dropped {len(DROP_META)} metadata/leakage columns")
print(f"Remaining: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Post-hoc assertion
POST_HOC = ["icu_los_days", "hospital_los_days", "icu_mortality"]
for col in POST_HOC:
    assert col not in df.columns, f"Leakage column still present: {col}"
print("Assertion PASS: no leakage columns in dataframe")

Dropped 6 metadata/leakage columns
Remaining: 11,164 rows x 72 columns
Assertion PASS: no leakage columns in dataframe


### Findings — Cell 2: Leakage and metadata columns dropped

---

| Column | Category | Reason |
|---|---|---|
| `icu_los_days` | **Leakage** | Known only at discharge; unavailable at ICU admission |
| `hospital_los_days` | **Leakage** | Known only at discharge; unavailable at ICU admission |
| `icu_mortality` | **Leakage** | Alternative outcome variable — not a predictor |
| `ethnicity` | Raw string | Information not used as a feature (no dummy encoding retained) |
| `apacheadmissiondx` | Raw string | Encoded in `dx_Sepsis_*` dummy columns (NB03 Cell 7) |
| `uniquepid` | Redundant ID | `patientunitstayid` is the primary patient key |

Post-hoc assertion: PASS — no leakage columns remain.

---
## Cell 3 — B2 Feature curation: remove redundant features

**Plan.** Remove three groups of redundant features identified in supervisor Issue B2,
then define the curated `MODELLING_COLS` (58 features for LR/RF baselines).

**(a) GCS components** — `eyes_miss`, `motor_miss`, `verbal_miss`
`gcs_total` = eyes + motor + verbal. The individual component missingness indicators
are redundant once the composite score is retained.

**(b) ABG redundancy** — `pao2`, `fio2_frac`, `pao2_miss`, `fio2_frac_miss`
`pf_ratio = pao2 / fio2_frac`. PaO₂ alone is clinically ambiguous without FiO₂
context. `pf_ratio` encodes both in the standard clinical form.

**(c) MAP redundancy** — `map_min`, `map_max`, `map_min_miss`, `map_max_miss`
`meanbp` (APACHE worst-in-24h) captures worst hypotensive episode;
`map_mean` captures typical 24h haemodynamic state. `map_min` overlaps with `meanbp`;
`map_max` adds minimal signal in sepsis mortality context.

In [3]:
# ── B2 drops ──────────────────────────────────────────────────────────────────
B2_DROP = [
    # (a) GCS component missingness indicators
    "eyes_miss", "motor_miss", "verbal_miss",
    # (b) ABG redundancy
    "pao2", "fio2_frac", "pao2_miss", "fio2_frac_miss",
    # (c) MAP redundancy
    "map_min", "map_max", "map_min_miss", "map_max_miss",
]
# Only drop those that exist
B2_DROP_EXISTING = [c for c in B2_DROP if c in df.columns]
df = df.drop(columns=B2_DROP_EXISTING)
print(f"B2 drops applied: {len(B2_DROP_EXISTING)} columns removed")
for col in B2_DROP_EXISTING:
    print(f"  - {col}")

# ── Define MODELLING_COLS ─────────────────────────────────────────────────────
ID_COLS     = ["patientunitstayid", "hospitalid"]
OUTCOME_COL = "hospital_mortality"

PRIMARY_CONTINUOUS = [
    "age_numeric", "heartrate", "meanbp", "respiratoryrate", "temperature",
    "pf_ratio", "wbc", "creatinine", "bilirubin", "bun", "glucose",
    "sodium", "ph", "hematocrit", "albumin", "gcs_total",
]
LAB_FEATURES  = ["potassium_max", "platelets_min", "lactate_max"]
MAP_FEATURES  = ["map_mean"]   # map_min and map_max dropped (B2c)
BINARY_FLAGS  = ["gender_male", "vent", "intubated", "dialysis", "vasopressor_24h"]
UNIT_DUMMIES  = [c for c in df.columns if c.startswith("unit_")]
DX_DUMMIES    = [c for c in df.columns if c.startswith("dx_")]
MISS_INDS     = [c for c in df.columns if c.endswith("_miss")]

MODELLING_COLS = (PRIMARY_CONTINUOUS + LAB_FEATURES + MAP_FEATURES +
                  BINARY_FLAGS + UNIT_DUMMIES + DX_DUMMIES + MISS_INDS)

# Validate all MODELLING_COLS exist in df
missing = [c for c in MODELLING_COLS if c not in df.columns]
assert not missing, f"MODELLING_COLS columns missing from df: {missing}"

all_feature_cols = [c for c in df.columns if c not in ID_COLS + [OUTCOME_COL]]
extra = set(MODELLING_COLS) - set(all_feature_cols)
unaccounted = set(all_feature_cols) - set(MODELLING_COLS)
assert not extra, f"MODELLING_COLS contains columns not in df: {extra}"
assert not unaccounted, f"df has feature columns not in MODELLING_COLS: {unaccounted}"

print(f"\nMODELLING_COLS: {len(MODELLING_COLS)} features")
print(f"  Primary continuous (excl. ABG components) : {len(PRIMARY_CONTINUOUS)}")
print(f"  Lab features                               : {len(LAB_FEATURES)}")
print(f"  MAP features (map_mean only)               : {len(MAP_FEATURES)}")
print(f"  Binary flags                               : {len(BINARY_FLAGS)}")
print(f"  Unit dummies                               : {len(UNIT_DUMMIES)}")
print(f"  Dx dummies                                 : {len(DX_DUMMIES)}")
print(f"  Missingness indicators                     : {len(MISS_INDS)}")

B2 drops applied: 11 columns removed
  - eyes_miss
  - motor_miss
  - verbal_miss
  - pao2
  - fio2_frac
  - pao2_miss
  - fio2_frac_miss
  - map_min
  - map_max
  - map_min_miss
  - map_max_miss

MODELLING_COLS: 58 features
  Primary continuous (excl. ABG components) : 16
  Lab features                               : 3
  MAP features (map_mean only)               : 1
  Binary flags                               : 5
  Unit dummies                               : 7
  Dx dummies                                 : 6
  Missingness indicators                     : 20


### Findings — Cell 3: B2 curation applied

---

**Columns removed (11 total):**

| Group | Removed | Retained instead |
|---|---|---|
| GCS components | `eyes_miss`, `motor_miss`, `verbal_miss` | `gcs_total` + `gcs_total_miss` |
| ABG redundancy | `pao2`, `fio2_frac`, `pao2_miss`, `fio2_frac_miss` | `pf_ratio` + `pf_ratio_miss` |
| MAP redundancy | `map_min`, `map_max`, `map_min_miss`, `map_max_miss` | `meanbp` + `map_mean` + `map_mean_miss` |

**MODELLING_COLS summary (58 features):**

| Group | Count |
|---|---:|
| Primary continuous (vitals + labs, excl. ABG components) | 16 |
| Lab features (potassium, platelets, lactate) | 3 |
| MAP features (map_mean only) | 1 |
| Binary flags | 5 |
| Unit ICU-type dummies | 7 |
| Admission dx dummies | 6 |
| Missingness indicators (`_miss`) | 20 |
| **Total MODELLING_COLS** | **58** |

All 58 columns verified present in dataframe. No leakage column in MODELLING_COLS (assertion passed).

---
## Cell 4 — GP terminal selection

**Plan.** Define `GP_TERMINALS` — the subset of `MODELLING_COLS` used by PySR symbolic
regression. GP works best with 20–30 continuous and binary terminals that have real
physiological signal. Two additional exclusions apply beyond B2:

- **Albumin (B6):** 39.6% missing → imputed at median 2.4 g/dL. With 39.6% of values
  identical, any GP expression using albumin would be partially driven by the imputed
  constant rather than true physiology. Retained in `MODELLING_COLS` for LR/RF
  (regularisation handles low-variance features); excluded from GP terminals.
- **Unit and dx dummies:** Categorical indicators, not continuous mathematical variables.
  GP symbolic expressions require features that can participate meaningfully in arithmetic
  — dummies add only binary branching that GP tree-based models already handle poorly.
- **25 residual `_miss` indicators:** MAR/MCAR features with <5 pp mortality gap.
  Their imputed values are valid and carry the real signal. Only two MNAR `_miss`
  indicators are included: `pf_ratio_miss` (+19.7 pp gap) and `lactate_max_miss`
  (+12.4 pp gap).

In [4]:
# ── GP_TERMINALS: 26 features ──────────────────────────────────────────────────
# PRIMARY_CONTINUOUS minus albumin (B6 exclusion)
GP_PRIMARY = [c for c in PRIMARY_CONTINUOUS if c != "albumin"]   # 15 features

GP_TERMINALS = (
    GP_PRIMARY        +   # 15 — physiology without albumin
    LAB_FEATURES      +   # 3  — potassium, platelets, lactate
    MAP_FEATURES      +   # 1  — map_mean
    BINARY_FLAGS      +   # 5  — vent, vasopressor, dialysis, gender, intubated
    ["pf_ratio_miss",     # MNAR: +19.7 pp mortality gap
     "lactate_max_miss"]  # MNAR: +12.4 pp mortality gap
)

assert all(t in MODELLING_COLS for t in GP_TERMINALS),     "GP_TERMINALS not a subset of MODELLING_COLS"

print(f"GP_TERMINALS: {len(GP_TERMINALS)} features")
print()
for i, t in enumerate(GP_TERMINALS, 1):
    excl = " [MNAR _miss]" if t.endswith("_miss") else ""
    print(f"  {i:>2}. {t}{excl}")

print(f"\nAlbumin excluded from GP_TERMINALS (retained in MODELLING_COLS):")
print(f"  albumin: 39.6% missing, imputed at 2.4 g/dL (B6 limitation)")

GP_TERMINALS: 26 features

   1. age_numeric
   2. heartrate
   3. meanbp
   4. respiratoryrate
   5. temperature
   6. pf_ratio
   7. wbc
   8. creatinine
   9. bilirubin
  10. bun
  11. glucose
  12. sodium
  13. ph
  14. hematocrit
  15. gcs_total
  16. potassium_max
  17. platelets_min
  18. lactate_max
  19. map_mean
  20. gender_male
  21. vent
  22. intubated
  23. dialysis
  24. vasopressor_24h
  25. pf_ratio_miss [MNAR _miss]
  26. lactate_max_miss [MNAR _miss]

Albumin excluded from GP_TERMINALS (retained in MODELLING_COLS):
  albumin: 39.6% missing, imputed at 2.4 g/dL (B6 limitation)


### Findings — Cell 4: GP terminal set

---

**GP_TERMINALS (26 features):**

| Group | Features | Count |
|---|---|---:|
| Primary continuous (no albumin) | age, HR, meanbp, RR, temp, pf_ratio, WBC, Cr, Bili, BUN, Glu, Na, pH, Hct, GCS | 15 |
| Lab features | potassium_max, platelets_min, lactate_max | 3 |
| MAP (mean only) | map_mean | 1 |
| Binary flags | gender_male, vent, intubated, dialysis, vasopressor_24h | 5 |
| MNAR indicators | pf_ratio_miss (+19.7 pp), lactate_max_miss (+12.4 pp) | 2 |
| **Total GP_TERMINALS** | | **26** |

**Albumin exclusion rationale (Issue B6):**
Albumin is missing in 39.6% of patients, all imputed at the population median (2.4 g/dL).
Including albumin in a GP expression risks the formula capturing the imputation artefact —
a constant value shared by 39.6% of patients — rather than true hepatic/nutritional physiology.
Albumin is retained in `MODELLING_COLS` for LR/RF baselines where regularisation naturally
down-weights low-variance features. `albumin_miss` (+4.7 pp MNAR gap) is available in
`MODELLING_COLS` but excluded from GP terminals given the weak MNAR signal.

---
## Cell 5 — Curation log and feature inventory

**Plan.** Build a complete record of every column decision: which columns were kept,
which were dropped, the reason, and the missingness tier. Save as `05_curation_log.csv`.

In [5]:
records = []

# Dropped columns (leakage + raw strings + B2)
dropped_leakage = {"icu_los_days":"leakage","hospital_los_days":"leakage","icu_mortality":"leakage"}
dropped_raw     = {"ethnicity":"raw_string","apacheadmissiondx":"raw_string","uniquepid":"redundant_id"}
dropped_b2      = {c:"B2_redundancy" for c in B2_DROP_EXISTING}

for col, reason in {**dropped_leakage,**dropped_raw,**dropped_b2}.items():
    records.append({"column":col,"decision":"DROPPED","reason":reason,
                    "in_MODELLING_COLS":False,"in_GP_TERMINALS":False,"missingness_tier":"—"})

# ID and outcome columns
for col in ID_COLS + [OUTCOME_COL]:
    records.append({"column":col,"decision":"METADATA","reason":"id_or_outcome",
                    "in_MODELLING_COLS":False,"in_GP_TERMINALS":False,"missingness_tier":"—"})

# Modelling columns
miss_cols = feat[[c for c in feat.columns if c.endswith("_miss")]].mean()*100

for col in MODELLING_COLS:
    in_gp = col in GP_TERMINALS
    # missingness tier from original _miss indicator
    miss_col = col + "_miss"
    if miss_col in feat.columns:
        pct = float(feat[miss_col].mean()*100)
        tier = ">50%" if pct>50 else "20-50%" if pct>=20 else "<20%"
    elif col.endswith("_miss"):
        tier = "binary_indicator"
    else:
        tier = "<1%"
    reason = "MODELLING_COLS"
    if col in GP_TERMINALS: reason += " + GP_TERMINAL"
    if col == "albumin": reason += " (B6: excluded from GP)"
    records.append({"column":col,"decision":"KEPT","reason":reason,
                    "in_MODELLING_COLS":True,"in_GP_TERMINALS":in_gp,"missingness_tier":tier})

import pandas as pd
log_df = pd.DataFrame(records)
log_path = TABLES / "05_curation_log.csv"
log_df.to_csv(log_path, index=False)

kept    = (log_df.decision=="KEPT").sum()
dropped = (log_df.decision=="DROPPED").sum()
gp_t    = log_df.in_GP_TERMINALS.sum()
print(f"Curation log: {len(log_df)} entries")
print(f"  KEPT    : {kept} columns ({gp_t} in GP_TERMINALS)")
print(f"  DROPPED : {dropped} columns")
print(f"Saved: {log_path}")

Curation log: 78 entries
  KEPT    : 58 columns (26 in GP_TERMINALS)
  DROPPED : 17 columns
Saved: C:\ML PROJECT\sepsis-gp\results\tables\05_curation_log.csv


### Findings — Cell 5: Curation log

---

| Decision | Count |
|---|---:|
| KEPT (in MODELLING_COLS) | 58 |
| — of which in GP_TERMINALS | 26 |
| DROPPED (leakage + raw + B2) | 17 |
| METADATA (ID + outcome) | 3 |
| **Total accounted** | **78** |

Saved to `results/tables/05_curation_log.csv`. Every column from `features_block3.parquet`
is accounted for with a documented decision and reason. This log is the audit trail for
all feature engineering and curation decisions made in NB03 and NB05.

---
## Cell 6 — Save features_curated.parquet and feature_config.json

**Plan.** Save the curated feature matrix and the feature configuration file.
`feature_config.json` stores `MODELLING_COLS` and `GP_TERMINALS` so all downstream
notebooks (NB07–NB14) import the same feature sets without re-defining them.

In [6]:
# ── Save parquet ──────────────────────────────────────────────────────────────
SAVE_COLS = ID_COLS + [OUTCOME_COL] + MODELLING_COLS
out_path  = DATA / "features_curated.parquet"
df[SAVE_COLS].to_parquet(out_path, index=False)

verify = pd.read_parquet(out_path)
assert verify.shape == (len(df), len(SAVE_COLS)), "Round-trip shape mismatch"
assert verify["hospital_mortality"].isna().sum() == 0, "NaN in outcome after save"
print(f"Saved: {out_path}")
print(f"  Rows    : {verify.shape[0]:,}")
print(f"  Columns : {verify.shape[1]}")
print(f"  = {len(ID_COLS)} metadata + 1 outcome + {len(MODELLING_COLS)} MODELLING_COLS")

# ── Save feature config ───────────────────────────────────────────────────────
config = {
    "MODELLING_COLS" : MODELLING_COLS,
    "GP_TERMINALS"   : GP_TERMINALS,
    "OUTCOME_COL"    : OUTCOME_COL,
    "ID_COLS"        : ID_COLS,
}
cfg_path = DATA / "feature_config.json"
with open(cfg_path, "w") as f:
    _json.dump(config, f, indent=2)
print(f"Saved: {cfg_path}")
print(f"  MODELLING_COLS : {len(MODELLING_COLS)}")
print(f"  GP_TERMINALS   : {len(GP_TERMINALS)}")
print()
print("NB05 complete. features_curated.parquet is the input for NB07–NB14.")

Saved: C:\ML PROJECT\sepsis-gp\data\processed\features_curated.parquet
  Rows    : 11,164
  Columns : 61
  = 2 metadata + 1 outcome + 58 MODELLING_COLS
Saved: C:\ML PROJECT\sepsis-gp\data\processed\feature_config.json
  MODELLING_COLS : 58
  GP_TERMINALS   : 26

NB05 complete. features_curated.parquet is the input for NB07–NB14.


### Findings — Cell 6: Outputs saved

---

| Output | File | Rows | Columns |
|---|---|---:|---:|
| Curated feature matrix | `data/processed/features_curated.parquet` | 11,164 | 61 |
| Feature configuration | `data/processed/feature_config.json` | — | — |
| Curation log | `results/tables/05_curation_log.csv` | 78 | 6 |

**Column breakdown of features_curated.parquet (61 total):**

| Group | Columns |
|---|---|
| ID | `patientunitstayid`, `hospitalid` |
| Outcome | `hospital_mortality` |
| MODELLING_COLS | 58 curated feature columns |

**feature_config.json contents:**
- `MODELLING_COLS`: 58 features (LR / RF baselines)
- `GP_TERMINALS`: 26 features (PySR symbolic regression)
- `OUTCOME_COL`: `hospital_mortality`
- `ID_COLS`: `patientunitstayid`, `hospitalid`

Round-trip verification: PASS — shape and 0 NaN outcome confirmed.

---

## NB05 — Summary

**Feature curation complete.**

| Item | Value |
|---|---|
| Input | `features_block3.parquet` (11,164 × 78) |
| Output | `features_curated.parquet` (11,164 × 61) |
| MODELLING_COLS | **58** features (LR / RF / XGBoost baselines) |
| GP_TERMINALS | **26** features (PySR symbolic regression) |
| Columns dropped | 11 redundant (B2) + 6 leakage/metadata = **17 total** |

**Next:** NB06 — APACHE-IV Baseline. Extract `predictedHospitalMortality` from
`apachePatientResult.csv.gz`, compute AUROC / Brier / ECE, and save as the
clinical severity score benchmark for all model comparisons.